### Obx events mapping as of 3/25/26
- 0 is analog visual
- 1 is analog audio
- 2 is trial start
- 3 is frames
- 4 is left port
- 5 is center port
- 6 is right port

In [4]:
from labdata.schema import (
    EphysRecording,
    SpikeSorting,
    Dataset,
    StreamSync,
)

from tqdm import tqdm

In [ ]:
# #for populating past nidq/obx events that haven't been populated already, run the following on hodgkin
# a = (EphysRecording).fetch("KEY")
# for k in a:
#     (EphysRecording & k).add_nidq_events()

### Populate the StreamSync table
Note: If a session doesn't have the DatasetEvents populated for the nidq/obx streams, this insertion will fail.

In [ ]:
sorted_sessions = (EphysRecording & SpikeSorting).proj()
key = sorted_sessions.fetch("subject_name", "session_name", as_dict=True)
all_sessions = (
    Dataset() & key
).proj()  # gets all ephys sessions with both chipmunk and ephys_g* dataset entries

for ses in tqdm(all_sessions):
    try:
        ephys_dset = (
            EphysRecording() & f'session_name = "{ses["session_name"]}"'
        ).fetch("dataset_name")[0]
    except Exception as e:
        print(
            LookupError(f"Couldn't find an ephys session for {ses}.\nException: {e}.")
        )
        continue

    if ses["subject_name"] == "GRB006" or "GRB041":
        # this mouse with recorded with a nidq, not a obx
        # requires an imec0 entry in DatasetEvents which is populated during spike sorting
        StreamSync().insert(
            [
                dict(
                    ses,
                    stream_name="imec0",
                    event_name="6",
                    clock_dataset=ephys_dset,
                    clock_stream="nidq",
                    clock_stream_event="0",  # this is where the bpod trial start is wired up to in the nidq
                )
            ],
            skip_duplicates=True,
        )
        continue

    if ses["dataset_name"] == "chipmunk":
        # for syncing bpod with obx
        StreamSync().insert(
            [
                dict(
                    ses,
                    stream_name="bpod",
                    event_name="sync",
                    clock_dataset=ephys_dset,
                    clock_stream="obx",
                    clock_stream_event="io2",  # this is where the bpod trial start is wired up to in the obx
                )
            ],
            skip_duplicates=True,
        )
    elif ses["dataset_name"] == ephys_dset:
        # for syncing probe with obx
        # requires an imec0 entry in DatasetEvents which is populated during spike sorting
        StreamSync().insert(
            [
                dict(
                    ses,
                    stream_name="imec0",
                    event_name="6",
                    clock_dataset=ses["dataset_name"],
                    clock_stream="obx",
                    clock_stream_event=6,
                )
            ],
            skip_duplicates=True,
        )
    else:
        print(
            NotImplementedError(
                f"Inserting a StreamSync row for dataset {ses['dataset_name']} is not yet supported."
            )
        )
        continue